# Anomaly Detection — Extended: Autoencoder + Deep Evaluation
## Toy Neural Network · LOF · ROC/PR Curves · Feature Importance · Model Persistence

This notebook extends the anomaly detection pipeline from notebooks **0** and **1** with:

1. **Local Outlier Factor** — a third classical detector for comparison
2. **Toy Neural Network Autoencoder** — built from scratch in PyTorch; learns to reconstruct normal signals and uses high reconstruction error as an anomaly score
3. **Rich evaluation** — ROC curves, PR curves, normalized confusion matrices for all three models side-by-side
4. **Hyperparameter sensitivity** — how does LOF's `n_neighbors` affect performance?
5. **Permutation feature importance** — which of our 5 features matters most?
6. **Model persistence** — save trained models and results to disk

### What you'll learn
- How autoencoders work and why they're natural anomaly detectors
- How to implement a PyTorch autoencoder and train it on normal data only
- How to interpret reconstruction error as an anomaly score
- How to produce publication-quality evaluation plots

### Prerequisites
Notebooks **0** or **1** (for context). Basic Python and NumPy. PyTorch for the autoencoder section.

---
*Algorithms: OC-SVM · Isolation Forest · LOF · Autoencoder (PyTorch)*

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings, os, hashlib
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
SEED = 42

plt.rcParams.update({
    'figure.dpi': 130, 'axes.titlesize': 13,
    'axes.labelsize': 11, 'font.size': 10, 'legend.fontsize': 9,
})
sns.set_style("whitegrid")

print(f"PyTorch {torch.__version__}")
print("All imports successful.")

In [ ]:
# ── Reproduce the same dataset + features from notebook 1 ────
fs = 10000; T_window = 0.5; N_samples = int(fs * T_window)
n_windows = 500; fault_ratio = 0.10
shaft_freq = 30; bpfi = 5.2 * shaft_freq
t = np.arange(N_samples) / fs

def generate_normal_vibration(t, shaft_freq, fs):
    sig = (1.0*np.sin(2*np.pi*shaft_freq*t) + 0.5*np.sin(2*np.pi*2*shaft_freq*t+0.3) +
           0.3*np.sin(2*np.pi*3*shaft_freq*t+1.1) + 0.15*np.sin(2*np.pi*4*shaft_freq*t+0.7))
    return sig + 0.4*np.random.randn(len(t))

def generate_faulty_vibration(t, shaft_freq, bpfi, fs):
    normal = generate_normal_vibration(t, shaft_freq, fs)
    impulse_period = 1.0 / bpfi
    impulses = np.zeros_like(t)
    for ti in np.arange(0, t[-1], impulse_period):
        idx = np.argmin(np.abs(t - ti))
        if idx < len(t) - 10:
            burst_len = min(150, len(t) - idx)
            burst = np.exp(-np.arange(burst_len)/20.0) * np.random.uniform(2.5, 4.5)
            carrier = np.sin(2*np.pi*3500*np.arange(burst_len)/fs)
            impulses[idx:idx+burst_len] += burst * carrier
    am = 1.0 + 0.3*np.sin(2*np.pi*bpfi*t)
    return normal * am + impulses

def extract_features(sig, fs):
    N = len(sig)
    rms = np.sqrt(np.mean(sig**2)); mean_abs = np.mean(np.abs(sig)); peak = np.max(np.abs(sig))
    freqs = np.fft.rfftfreq(N, 1/fs); fft_mag = np.abs(np.fft.rfft(sig))
    return [stats.kurtosis(sig, fisher=False), peak/rms if rms>0 else 0,
            stats.skew(sig), rms/mean_abs if mean_abs>0 else 0,
            np.sum(freqs*fft_mag)/np.sum(fft_mag) if np.sum(fft_mag)>0 else 0]

n_faulty = int(n_windows*fault_ratio); n_normal = n_windows - n_faulty
signals, labels = [], []
for _ in range(n_normal): signals.append(generate_normal_vibration(t, shaft_freq, fs)); labels.append(0)
for _ in range(n_faulty): signals.append(generate_faulty_vibration(t, shaft_freq, bpfi, fs)); labels.append(1)
signals, labels = np.array(signals), np.array(labels)
idx = np.random.permutation(n_windows); signals, labels = signals[idx], labels[idx]

feature_names = ['Kurtosis', 'Crest Factor', 'Skewness', 'Shape Factor', 'Spectral Centroid']
df = pd.DataFrame([extract_features(signals[i], fs) for i in range(n_windows)], columns=feature_names)
df['Label'] = labels; df['Label_str'] = df['Label'].map({0:'Normal', 1:'Fault'})

X = df[feature_names].values; y_true = df['Label'].values
scaler = StandardScaler(); X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2); X_pca = pca.fit_transform(X_scaled)

# Re-train baseline models
ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.10).fit(X_scaled)
y_pred_ocsvm = (ocsvm.predict(X_scaled) == -1).astype(int)
scores_ocsvm = ocsvm.decision_function(X_scaled)

iforest = IsolationForest(n_estimators=200, contamination=0.10, random_state=SEED).fit(X_scaled)
y_pred_iforest = (iforest.predict(X_scaled) == -1).astype(int)
scores_iforest = iforest.decision_function(X_scaled)

print(f"Dataset ready: {n_windows} windows | Normal: {(labels==0).sum()} | Fault: {(labels==1).sum()}")

## §1 — Local Outlier Factor (LOF)

**LOF** (Breunig et al., 2000) is a **density-based** anomaly detector. For each point, it compares the local density of that point's neighbourhood to the local density of its neighbours. Points in sparse regions relative to their neighbours get a **high LOF score** → anomaly.

**Key hyperparameter: `n_neighbors`** — the size of the local neighbourhood. Too small → noisy; too large → misses local density variations. We tune it in §8.

In [ ]:
# ── §1 — Train Local Outlier Factor ──────────────────────────
# novelty=True is required when we call predict() on training data
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.10, novelty=True)
lof.fit(X_scaled)

y_pred_lof = (lof.predict(X_scaled) == -1).astype(int)
scores_lof = lof.decision_function(X_scaled)  # more negative = more anomalous

print("=== Local Outlier Factor Results ===")
print(f"n_neighbors = 20 | contamination = 0.10")
print(f"\nPredicted Normal: {(y_pred_lof == 0).sum()}")
print(f"Predicted Fault:  {(y_pred_lof == 1).sum()}")
print(f"\n{classification_report(y_true, y_pred_lof, target_names=['Normal', 'Fault'])}")

## §2 — Toy Neural Network: Autoencoder

An **autoencoder** is a neural network trained to **compress** its input to a low-dimensional *latent code*, then **reconstruct** the original input from that code.

```
Input (5)  →  Encoder  →  Latent (2)  →  Decoder  →  Output (5)
```

When trained **only on normal data**, the network learns to reconstruct normal patterns well. When it encounters a **fault** pattern it has never seen, reconstruction is poor → **high MSE error → anomaly detected**.

This is unsupervised anomaly detection with a neural network — no fault labels required during training.

### Architecture

| Layer | Type | Input → Output | Activation |
|---|---|---|---|
| enc1 | Linear | 5 → 16 | ReLU |
| enc2 | Linear | 16 → 8 | ReLU |
| enc3 | Linear | 8 → 2 | — (latent) |
| dec1 | Linear | 2 → 8 | ReLU |
| dec2 | Linear | 8 → 16 | ReLU |
| dec3 | Linear | 16 → 5 | — (output) |

In [ ]:
# ── §2 — Define the Autoencoder ──────────────────────────────
class Autoencoder(nn.Module):
    """
    Toy autoencoder for anomaly detection.
    Trained only on normal data; faults have high reconstruction error.
    """
    def __init__(self, input_dim=5, latent_dim=2):
        super().__init__()

        # Encoder: compress input down to latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),  # 5 → 16
            nn.ReLU(),
            nn.Linear(16, 8),           # 16 → 8
            nn.ReLU(),
            nn.Linear(8, latent_dim),   # 8 → 2  (bottleneck)
        )

        # Decoder: reconstruct original features from latent code
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),   # 2 → 8
            nn.ReLU(),
            nn.Linear(8, 16),           # 8 → 16
            nn.ReLU(),
            nn.Linear(16, input_dim),   # 16 → 5  (reconstruct)
        )

    def forward(self, x):
        z = self.encoder(x)   # compress
        x_hat = self.decoder(z)  # reconstruct
        return x_hat

    def encode(self, x):
        """Return the latent representation."""
        return self.encoder(x)


model = Autoencoder(input_dim=5, latent_dim=2)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## §3 — Train Autoencoder on Normal Data Only

**Key training rule:** we only feed **normal windows** into the autoencoder during training. The network never sees a fault, so it optimises entirely for reconstructing normal patterns.

We use:
- **Loss**: Mean Squared Error (MSE) — penalises reconstruction deviation
- **Optimiser**: Adam (adaptive learning rate)
- **Epochs**: 200

In [ ]:
# ── §3 — Training Loop ───────────────────────────────────────

# Use only NORMAL windows for training
X_normal = X_scaled[y_true == 0]
X_tensor = torch.FloatTensor(X_normal)

# DataLoader for mini-batch training
train_dataset = TensorDataset(X_tensor, X_tensor)  # input == target (autoencoder)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Re-instantiate a fresh model and set up training
ae_model = Autoencoder(input_dim=5, latent_dim=2)
criterion = nn.MSELoss()
optimizer = optim.Adam(ae_model.parameters(), lr=1e-3)

EPOCHS = 200
train_losses = []

ae_model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for X_batch, _ in train_loader:
        optimizer.zero_grad()
        X_recon = ae_model(X_batch)          # forward pass
        loss = criterion(X_recon, X_batch)   # MSE: how well did we reconstruct?
        loss.backward()                       # backpropagation: compute gradients
        optimizer.step()                      # update weights
        epoch_loss += loss.item() * len(X_batch)
    train_losses.append(epoch_loss / len(X_normal))

# ── Plot training loss curve ──
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, color='steelblue', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('§3 — Autoencoder Training Loss (Normal Data Only)', fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Final training loss: {train_losses[-1]:.6f}")
print("✅ The loss decreasing to a low value confirms the network learned to reconstruct normal signals.")

## §4 — Reconstruction Error as Anomaly Score

After training, we pass **all** windows (normal + fault) through the autoencoder and compute the per-sample **mean squared reconstruction error (MSRE)**:

$$\text{MSRE}_i = \frac{1}{d} \sum_{j=1}^{d} (\hat{x}_{ij} - x_{ij})^2$$

**Intuition:**
- Normal samples → low error (the model has seen this pattern)
- Fault samples → high error (the model never saw this during training)

We set the **anomaly threshold** at the 90th percentile of MSRE on **training** (normal-only) data — this directly corresponds to the 10% contamination rate.

In [ ]:
# ── §4 — Reconstruction error on ALL windows ─────────────────
ae_model.eval()
with torch.no_grad():
    X_all_tensor = torch.FloatTensor(X_scaled)
    X_recon_all  = ae_model(X_all_tensor).numpy()

# Per-sample MSE reconstruction error
recon_error = np.mean((X_scaled - X_recon_all) ** 2, axis=1)

# Threshold: 90th percentile of normal training errors
normal_errors = recon_error[y_true == 0]
threshold_ae  = np.percentile(normal_errors, 90)

# Classify: above threshold = fault
y_pred_ae     = (recon_error > threshold_ae).astype(int)
scores_ae     = recon_error   # higher = more anomalous

print(f"Anomaly threshold (90th pct of normal errors): {threshold_ae:.6f}")
print(f"\nPredicted Normal: {(y_pred_ae == 0).sum()}")
print(f"Predicted Fault:  {(y_pred_ae == 1).sum()}")
print(f"\n{classification_report(y_true, y_pred_ae, target_names=['Normal', 'Fault'])}")

# ── Plot error distributions ──
fig, ax = plt.subplots(figsize=(10, 4))
for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'crimson', 'Fault')]:
    ax.hist(recon_error[y_true == label], bins=40, alpha=0.6,
            color=color, label=name, edgecolor='white')
ax.axvline(threshold_ae, color='black', linestyle='--', linewidth=2, label=f'Threshold = {threshold_ae:.4f}')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Count')
ax.set_title('§4 — Autoencoder Reconstruction Error: Normal vs. Fault', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("\n🔍 Fault windows (red) have systematically HIGHER reconstruction errors.")
print("   The dashed threshold separates the two distributions.")

## §5 — Three-Model Comparison: OC-SVM · Isolation Forest · Autoencoder

Let's compare all three models on the same held-out data with a unified metrics table and bar chart.

In [ ]:
# ── §5 — Unified Metrics Table ────────────────────────────────
all_models = {
    'OC-SVM':           (y_pred_ocsvm,  scores_ocsvm),
    'Isolation Forest': (y_pred_iforest, scores_iforest),
    'LOF':              (y_pred_lof,     scores_lof),
    'Autoencoder':      (y_pred_ae,      scores_ae),
}

rows = []
for name, (y_pred, scores) in all_models.items():
    # For models where lower score = more anomalous, negate so higher = anomalous
    flip = -1 if name != 'Autoencoder' else 1
    rows.append({
        'Model':     name,
        'Precision': precision_score(y_true, y_pred),
        'Recall':    recall_score(y_true, y_pred),
        'F1':        f1_score(y_true, y_pred),
        'Accuracy':  accuracy_score(y_true, y_pred),
        'ROC-AUC':   roc_auc_score(y_true, flip * scores),
        'PR-AUC':    average_precision_score(y_true, flip * scores),
    })

summary_df = pd.DataFrame(rows).set_index('Model').round(3)
print("=== Model Comparison ===")
print(summary_df.to_string())

# ── Bar chart ──
metrics = ['Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']
x = np.arange(len(metrics)); width = 0.2
palette = ['steelblue', 'crimson', 'darkorange', 'seagreen']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, row) in enumerate(summary_df[metrics].iterrows()):
    ax.bar(x + i * width, row.values, width, label=name, color=palette[i], alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('§5 — Four-Model Comparison: Key Metrics', fontweight='bold')
ax.legend(loc='lower right')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout(); plt.show()

## §6 — ROC & Precision-Recall Curves (All Models)

A single threshold (e.g. contamination=0.10) only shows one operating point. The **ROC curve** and **PR curve** show performance **across all possible thresholds**:

- **ROC curve**: True Positive Rate vs. False Positive Rate. Area under the curve (AUC) = 1.0 is perfect. AUC = 0.5 = random.
- **PR curve**: Precision vs. Recall. More informative when classes are imbalanced (as here, 90/10 split). Higher area = better.

In [ ]:
# ── §6 — ROC & PR Curves ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (y_pred, scores), color in zip(
        all_models.keys(), all_models.values(), palette):
    flip = -1 if name != 'Autoencoder' else 1
    anomaly_scores = flip * scores

    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, anomaly_scores)
    auc_roc = roc_auc_score(y_true, anomaly_scores)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc_roc:.2f})')

    # PR curve
    prec, rec, _ = precision_recall_curve(y_true, anomaly_scores)
    ap = average_precision_score(y_true, anomaly_scores)
    axes[1].plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP={ap:.2f})')

# ROC random baseline
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('§6 — ROC Curves', fontweight='bold'); axes[0].legend(fontsize=8)

# PR baseline = fault fraction
baseline = y_true.mean()
axes[1].axhline(baseline, color='k', linestyle='--', alpha=0.4, label=f'Random ({baseline:.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('§6 — Precision-Recall Curves', fontweight='bold'); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## §7 — Confusion Matrix Grid (All Models)

A confusion matrix shows the **breakdown of predictions** into four quadrants:

|  | Predicted Normal | Predicted Fault |
|---|---|---|
| **Actual Normal** | True Negative (TN) ✅ | False Alarm (FP) ⚠️ |
| **Actual Fault** | Missed Fault (FN) ❌ | Detected Fault (TP) ✅ |

In industrial fault detection, **False Negatives (missed faults) are the most dangerous**.

In [ ]:
# ── §7 — Normalised Confusion Matrices ───────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, (y_pred, _)), color in zip(axes, all_models.items(), palette):
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=['Normal', 'Fault'],
        normalize='true',          # show proportions, not raw counts
        colorbar=False,
        ax=ax,
        cmap='Blues',
    )
    ax.set_title(f'{name}', fontweight='bold')

plt.suptitle('§7 — Normalised Confusion Matrices (row = actual, col = predicted)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("🔍 Bottom-left cell (row=Fault, col=Normal) = Missed Faults — keep this as low as possible.")

## §8 — Hyperparameter Sensitivity: LOF `n_neighbors`

How sensitive is LOF to the choice of `n_neighbors`? We sweep over several values and track both F1 and ROC-AUC to identify the stable operating range.

In [ ]:
# ── §8 — LOF n_neighbors sensitivity ─────────────────────────
neighbors_range = [3, 5, 10, 15, 20, 30, 50, 80]
f1_scores_lof, roc_aucs_lof = [], []

for k in neighbors_range:
    m = LocalOutlierFactor(n_neighbors=k, contamination=0.10, novelty=True)
    m.fit(X_scaled)
    yp = (m.predict(X_scaled) == -1).astype(int)
    sc = m.decision_function(X_scaled)
    f1_scores_lof.append(f1_score(y_true, yp))
    roc_aucs_lof.append(roc_auc_score(y_true, -sc))

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(neighbors_range, f1_scores_lof, 'o-', color='steelblue', linewidth=2, label='F1')
ax2.plot(neighbors_range, roc_aucs_lof, 's--', color='crimson', linewidth=2, label='ROC-AUC')

ax1.set_xlabel('n_neighbors'); ax1.set_ylabel('F1-score', color='steelblue')
ax2.set_ylabel('ROC-AUC', color='crimson')
ax1.set_title('§8 — LOF Sensitivity to n_neighbors', fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
ax1.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

best_k = neighbors_range[np.argmax(f1_scores_lof)]
print(f"Best n_neighbors by F1: {best_k}  (F1={max(f1_scores_lof):.3f})")

## §9 — Feature Importance via Permutation

**Permutation importance** answers: *"How much does model performance drop if I destroy the information in feature X?"*

For each feature, we shuffle its values randomly (breaking its relationship with the label) and measure the F1-score drop. A large drop = that feature is important. This works for **any** trained model — no access to model internals needed.

In [ ]:
# ── §9 — Permutation Feature Importance ──────────────────────
def permutation_importance_f1(model_name, predict_fn, X, y, feature_names, n_repeats=5):
    """Measure F1 drop when each feature is shuffled."""
    baseline_f1 = f1_score(y, predict_fn(X))
    importances = []
    for col in range(X.shape[1]):
        drops = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            X_perm[:, col] = np.random.permutation(X_perm[:, col])  # shuffle one feature
            f1_perm = f1_score(y, predict_fn(X_perm))
            drops.append(baseline_f1 - f1_perm)
        importances.append(np.mean(drops))
    return np.array(importances)

def ocsvm_predict(X):
    return (ocsvm.predict(X) == -1).astype(int)

def iforest_predict(X):
    return (iforest.predict(X) == -1).astype(int)

def lof_predict(X):
    return (lof.predict(X) == -1).astype(int)

def ae_predict(X):
    with torch.no_grad():
        recon = ae_model(torch.FloatTensor(X)).numpy()
    err = np.mean((X - recon) ** 2, axis=1)
    return (err > threshold_ae).astype(int)

imp_ocsvm   = permutation_importance_f1('OC-SVM',   ocsvm_predict,   X_scaled, y_true, feature_names)
imp_iforest = permutation_importance_f1('IForest',  iforest_predict, X_scaled, y_true, feature_names)
imp_lof     = permutation_importance_f1('LOF',      lof_predict,     X_scaled, y_true, feature_names)
imp_ae      = permutation_importance_f1('AE',       ae_predict,      X_scaled, y_true, feature_names)

# ── Plot ──
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, name, imp, color in zip(axes,
        ['OC-SVM', 'Isolation Forest', 'LOF', 'Autoencoder'],
        [imp_ocsvm, imp_iforest, imp_lof, imp_ae], palette):
    order = np.argsort(imp)[::-1]
    ax.barh([feature_names[i] for i in order], imp[order], color=color, alpha=0.85, edgecolor='white')
    ax.set_title(name, fontweight='bold')
    ax.axvline(0, color='gray', linewidth=0.8)
    ax.set_xlabel('F1 drop')

plt.suptitle('§9 — Permutation Feature Importance (higher = more important)', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()
print("🔍 Kurtosis and Crest Factor typically dominate — they capture the impulsive nature of faults.")

## §10 — Export Results & Save Models

Reproducibility matters. We save:
- The metrics comparison table as **CSV** (human-readable, shareable)
- OC-SVM and Isolation Forest with **joblib** (sklearn's recommended serializer)
- Autoencoder weights with **`torch.save`** (state dict — the PyTorch standard)
- SHA-256 checksums so you can verify the files haven't been modified

In [ ]:
# ── §10 — Save Results & Models ──────────────────────────────
import joblib

os.makedirs('results', exist_ok=True)

# 1. Save metrics table
csv_path = 'results/model_comparison.csv'
summary_df.to_csv(csv_path)
print(f"✅ Saved metrics table → {csv_path}")

# 2. Save sklearn models
joblib.dump(ocsvm,    'results/ocsvm.joblib')
joblib.dump(iforest,  'results/iforest.joblib')
joblib.dump(lof,      'results/lof.joblib')
print("✅ Saved OC-SVM, Isolation Forest, LOF → results/")

# 3. Save autoencoder weights
ae_path = 'results/autoencoder.pt'
torch.save(ae_model.state_dict(), ae_path)
print(f"✅ Saved Autoencoder state dict → {ae_path}")

# 4. SHA-256 checksums for reproducibility
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()[:16]  # first 16 chars for readability

print("\n--- File Sizes & Checksums ---")
for path in [csv_path, 'results/ocsvm.joblib', 'results/iforest.joblib',
             'results/lof.joblib', ae_path]:
    size = os.path.getsize(path)
    print(f"  {path:<35} {size:>8} bytes   sha256: {sha256(path)}")

print("\n✅ All models and results saved. To reload:")
print("   ocsvm = joblib.load('results/ocsvm.joblib')")
print("   ae_model.load_state_dict(torch.load('results/autoencoder.pt'))")